<img src="https://upload.wikimedia.org/wikipedia/commons/8/89/TheNorthFace_logo.svg" alt="THE NORTH FACE LOGO" width="50%" />

# The North Face e-commerce : boosting online sales !

## Company's Description 📇

The North Face is an American outdoor recreation product company, founded in 1968 to supply climbers. The North Face produces clothing, footwear, and outdoor equipment. By the late 1990s customers had expanded beyond outdoor enthusiasts and in the 2000s it became a fashion style symbol.


## Project 🚧

The marketing department would like to take advantage of machine learning solutions to boost online sales on the website : https://www.thenorthface.fr/

They have identified two major solutions that could have a huge effect on the conversion rates :
- Deploying a **recommender system** that will allow to suggest additionnal products to users, that are similar to the items they are already interested in. The recommendations could be materialized by a "you might also be interested by these products..." section that would appear on each product page of the website.
- Improving the structure of the products catalog thanks to **topic extraction**. The idea is to use unsupervised methods to challenge the existing categories : is it possible to find new categories of product that would be more suitable for the navigation on the website ?

## Goals 🎯

The project can be cut into three steps :

1. Identify groups of products that have similar descriptions.

2. Use the groups of similar products to build a simple recommender system algorithm.

3. Use topic modeling algorithms to automatically assess the latent topics present in the item descriptions.

## Scope of this project 🖼️

In this project, you'll work with a corpus of item descriptions from The North Face's product catalog. Data can be found here: 

👉👉<a href="https://www.kaggle.com/cclark/product-item-data?select=sample-data.csv" target="_blank"> The North Face product catalog</a> 👈👈


## Helpers 🦮

To help you achieve this project, here are a few tips that should help you: 

### Preprocessing of textual data

- First, you'll have to make some preprocessings to clean the corpus (in particular, don't forget to handle stop words and to lemmatize the documents). For this step, you'll need `pandas` and `spacy`.

- Then, you can to encode the texts with **TF-IDF transformation** (this can be done with scikit learn's `TFIDFVectorizer`)

### Part 1 : Groups of products with similar descriptions

For this part, you'll have to train a **clustering model** that will create groups of products for which the descriptions are "close" to each other. 
- You can use `DBSCAN` to make some clustering on the TF-IDF matrix. When dealing with texts, the distance metric to be used is "cosine" instead of "euclidean". You have to find some values for `eps` and `min_samples` that allow to get 10-20 clusters and not too many outliers.
- You can display a `wordcloud` for each cluster to analyze the results and see if the groups contain different words.

### Part 2 - Recommender system

Then, you can use the cluster ids from part 1 to build a **recommender system**. The aim is to be able to suggest to a user some products that are similar to the ones he is interested in. To do this, we will consider that products belonging to the same cluster are similar.

- Create a function named `find_similar_items` that takes an argument `item_id` representing the id of a product, and that returns a list of 5 item ids that belong to the same cluster as the product passed in argument
- Use python's `input()` function to allow the user to choose a product and perform some suggestions of similar items

### Part 3 : Topic modeling

*This part is independant from the two others.*

The aim is to use an LSA model to automatically extract latent topics in the products' descriptions.

- Use TruncatedSVD to make some topic extraction from the TF-IDF matrix. You can try values for `n_components` that allow to get 10-20 topics. Save the encoded matrix into a variable named `topic_encoded_df`.
- Contrary to clustering, LSA allows to map each document to a mixing of several topics. For this reason, it's a bit more difficult to interpret the topics as categories : one document can actually be related to several topics at a time. To make things easier, consider extracting the main topic of each document.
- As in part 1, you can display `wordclouds` to analyze the results.

## Deliverable 📬

To complete this project, your team should: 

- Train at least one **clustering model** on the corpus and display wordclouds describing the clusters
- Develop a simple python code that allows a user to type the id of a product he's interested in, and then get a list of similar items
- Train at least one **TruncatedSVD model** on the corpus and display wordclouds describing the latent topics


# 0. Load and clean, Lemmatize, vectorize data 
### Preprocessing of textual data

- First, catalog is loaded into a dataframe
- some preprocessings are made to clean the corpus :
  - separate name, separate the different paragraphs : description, details, fabric, weight, origin 
  - lower letters, 
  - handle stop words and lemmatize the documents 
For this step, `pandas` and `spacy` are used.

- Then, text is encoded with **TF-IDF transformation** (this can be done with scikit learn's `TFIDFVectorizer`)

In [5]:
# Load data 
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("cclark/product-item-data")

# print("Path to dataset files:", path)




In [6]:
import pandas as pd
import spacy

#data = path
dataset_init = pd.read_csv("sample-data.csv",encoding='utf-8')
dataset_init.head()

,id,description
0,1,Active classic boxers - There's a reason why o...
1,2,Active sport boxer briefs - Skinning up Glory ...
2,3,Active sport briefs - These superbreathable no...
3,4,"Alpine guide pants - Skin in, climb ice, switc..."
4,5,"Alpine wind jkt - On high ridges, steep ice an..."


In [7]:
dataset_init["description_"] = dataset_init["description"]

In [8]:
dataset = dataset_init.copy()

In [9]:

dataset["name"] = dataset["description"].astype("string").str.split(" - ", n=1).str[0].str.strip()
dataset["description"] = dataset["description"].astype("string").str.split(" - ", n=1).str[1].str.strip()
dataset["category"] = dataset["name"].astype("string").str.split(" ",n=1).str[0].str.strip()
dataset["article"] = dataset["name"].astype("string").str.split(" ",n=1).str[1].str.strip()
#Details
dataset["Details"] = dataset["description"].astype("string").str.split('Details:</b><ul> <li>',n=1).str[1]
#Fabric
dataset.head()

,id,description,description_,name,category,article,Details
0,1,There's a reason why our boxers are a cult fav...,Active classic boxers - There's a reason why o...,Active classic boxers,Active,classic boxers,"""Silky Capilene 1 fabric is ultralight, breath..."
1,2,Skinning up Glory requires enough movement wit...,Active sport boxer briefs - Skinning up Glory ...,Active sport boxer briefs,Active,sport boxer briefs,"""Stretch mesh provides support, open-weave mes..."
2,3,These superbreathable no-fly briefs are the mi...,Active sport briefs - These superbreathable no...,Active sport briefs,Active,sport briefs,"""Stretch mesh provides support, open-weave mes..."
3,4,"Skin in, climb ice, switch to rock, traverse a...","Alpine guide pants - Skin in, climb ice, switc...",Alpine guide pants,Alpine,guide pants,Durable stretch-woven polyester (47% recycled)...
4,5,"On high ridges, steep ice and anything alpine,...","Alpine wind jkt - On high ridges, steep ice an...",Alpine wind jkt,Alpine,wind jkt,"""Lightweight, breathable polyester ripstop fab..."


In [12]:
dataset.sample(12)

,id,description,description_,name,category,article,Details
13,14,June nights camped out in New Mexico's Navajo ...,Better sweater jkt - June nights camped out in...,Better sweater jkt,Better,sweater jkt,"""Soft fleece has a sweater-knit face, fleeced ..."
400,401,"Travel broadly and surf the world, but don't f...",Reversible phone home - Travel broadly and sur...,Reversible phone home,Reversible,phone home,Organic cotton fleece hoody lined with organic...
65,66,The roomy kangaroo-style front pocket on the K...,Synch marsupial - The roomy kangaroo-style fro...,Synch marsupial,Synch,marsupial,"""Made from soft, recycled double-faced fleece""..."
499,500,Time to simplify? Our All-Wear shorts prove th...,All-wear shorts - Time to simplify? Our All-We...,All-wear shorts,All-wear,shorts,Lightweight yet durable organic cotton</li> <l...
22,23,Designed for wet-then-chill pursuits like back...,Cap 2 zip neck - Designed for wet-then-chill p...,Cap 2 zip neck,Cap,2 zip neck,Capilene 2 fabric has open knit that invites a...
277,278,"After a day of Rivermouth's glassy offshore, s...",Corinne dress - After a day of Rivermouth's gl...,Corinne dress,Corinne,dress,"""Stretchy, soft, synthetic jersey knit""</li> <..."
320,321,When a shirt isn't enough and a jacket is too ...,Synch vest - When a shirt isn't enough and a j...,Synch vest,Synch,vest,"""Soft, recycled double-sided polyester fleece""..."
452,453,With cargo pockets pulled from vintage army bl...,Compound cargo shorts - With cargo pockets pul...,Compound cargo shorts,Compound,cargo shorts,"""Updated, regular fit""</li> <li>Grown-on waist..."
160,161,The highly adaptable Aravis Jacket eases the t...,Aravis jkt - The highly adaptable Aravis Jacke...,Aravis jkt,Aravis,jkt,Made of cozy soft polyester fleece</li> <li>Fu...
266,267,"An emerging classic, inspired by the original ...","Storm light jkt - An emerging classic, inspire...",Storm light jkt,Storm,light jkt,2-layer polyester fabric with a waterproof/bre...


In [67]:
#import plus musclé avec Claude Code : 
import html
import re
import pandas as pd

RE_NOM     = re.compile(r"^\s*(.*?)\s-\s(?=[A-Z0-9'])", re.DOTALL)
RE_INTRO   = re.compile(r"^(.*?)(?:<br><br><b>Details:</b>|$)", re.DOTALL)
RE_DETAIL  = re.compile(r"<li>(.*?)</li>", re.DOTALL)
RE_TISSU   = re.compile(r"<b>Fabric:\s*</b>(.*?)(?:<br><br>|$)", re.DOTALL)
# Accepte "(340 g 11.8 oz)" et "99 g (3.5 oz)"
RE_POIDS   = re.compile(r"<b>Weight:\s*</b>\(?\s*(\d+(?:\.\d+)?)\s*g\s*\(?\s*(\d+(?:\.\d+)?)\s*oz\s*\)")
RE_ORIGINE = re.compile(r"(?:Made|Printed)\s+in\s+([A-Za-z][A-Za-z .]*?)\.?\s*$")
RE_BALISES = re.compile(r"<[^>]+>")                       # (1) à ajouter avec les autres regex


def _nettoyer(s: str) -> str:
    """Retire les guillemets englobants, les balises HTML, dédouble les guillemets et décode les entités."""
    s = s.strip()
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
        s = s[1:-1]
    s = RE_BALISES.sub(" ", s)                             # (2) balises -> espace (avant unescape)
    s = html.unescape(s.replace('""', '"'))
    return re.sub(r"\s+", " ", s).strip()

def separer_nom(desc: str, ecart_max: int = 15) -> tuple[str | None, str]:
    """Renvoie (nom, reste). Garde le 2e ' - ' dans le nom s'il est à <= ecart_max caractères du 1er."""
    d = desc.strip()
    pos = [m.start() for m in re.finditer(r" - ", d)]
    if not pos:
        return None, d
    fin = pos[1] if len(pos) > 1 and pos[1] - pos[0] <= ecart_max else pos[0]
    return d[:fin].strip(), d[fin + 3:].strip()


def parser_description(desc) -> dict:
    vide = {"name": None, "text": None, "details": [], "fabric": None,
            "weight_g": None, "weight_oz": None, "origin": None}
    if not isinstance(desc, str) or not desc.strip():
        return vide
    desc = desc.strip()
    # m_nom = RE_NOM.match(desc)
    # reste = desc[m_nom.end():] if m_nom else desc
    name, reste = separer_nom(desc)
    
    m_poids, m_tissu, m_orig = RE_POIDS.search(reste), RE_TISSU.search(reste), RE_ORIGINE.search(reste)
    return {
        "name": name.lower() if name else None,
        "text": _nettoyer(RE_INTRO.match(reste).group(1)).lower(), # lower() APRÈS l'extraction ; plus de replace('""','"') : _nettoyer le fait déjà
        
        "details": [_nettoyer(x).lower() for x in RE_DETAIL.findall(reste)],
        "fabric": _nettoyer(m_tissu.group(1)) if m_tissu else None,
        "weight_g": float(m_poids.group(1)) if m_poids else None,
        "weight_oz": float(m_poids.group(2)) if m_poids else None,
        "origin": m_orig.group(1).strip() if m_orig else None,
    }


def charger_produits(chemin: str) -> pd.DataFrame:
    brut = pd.read_csv(chemin, header=0, names=["id", "description"], encoding="utf-8")
    parsed = pd.DataFrame([parser_description(x) for x in brut["description"]], index=brut.index)
    out = pd.concat([brut[["id"]], parsed], axis=1)
    # Les posters n'ont pas de poids : on ne signale que les autres champs
    manquants = out[out[["name", "fabric", "origin"]].isna().any(axis=1)]
    if not manquants.empty:
        print(f"{len(manquants)} ligne(s) incomplète(s) : id = {manquants['id'].tolist()}")
    return out


dataset = charger_produits("sample-data.csv")
dataset["details_txt"] = dataset["details"].str.join(" | ")   # export CSV

In [68]:
dataset

,id,name,text,details,fabric,weight_g,weight_oz,origin,details_txt
0,1,active classic boxers,there's a reason why our boxers are a cult fav...,"[silky capilene 1 fabric is ultralight, breath...",3.7-oz 100% all-recycled polyester with Gladio...,99.0,3.5,Mexico,"silky capilene 1 fabric is ultralight, breatha..."
1,2,active sport boxer briefs,skinning up glory requires enough movement wit...,"[stretch mesh provides support, open-weave mes...",4.6-oz 93% polyester (71% recycled)/7% spandex...,60.0,2.1,Israel,"stretch mesh provides support, open-weave mesh..."
2,3,active sport briefs,these superbreathable no-fly briefs are the mi...,"[stretch mesh provides support, open-weave mes...",4.6-oz 93% polyester (71% recycled)/7% spandex...,49.0,1.7,Israel,"stretch mesh provides support, open-weave mesh..."
3,4,alpine guide pants,"skin in, climb ice, switch to rock, traverse a...",[durable stretch-woven polyester (47% recycled...,7.6-oz 90-denier 92% polyester (47% recycled)/...,597.0,20.7,Vietnam,durable stretch-woven polyester (47% recycled)...
4,5,alpine wind jkt,"on high ridges, steep ice and anything alpine,...","[lightweight, breathable polyester ripstop fab...",Shell: 1.3-oz 20-denier 100% polyester ripstop...,331.0,11.5,China,"lightweight, breathable polyester ripstop fabr..."
...,...,...,...,...,...,...,...,...,...
495,496,cap 2 bottoms,cut loose from the maddening crowds and search...,[capilene 2 fabric has an open knit that invit...,4-oz 100% polyester (54% recycled) with Gladio...,155.0,5.5,Mexico,capilene 2 fabric has an open knit that invite...
496,497,cap 2 crew,this crew takes the edge off fickle weather. i...,[capilene 2 fabric has an open knit that invit...,4-oz 100% polyester (54% recycled) with Gladio...,170.0,6.0,Mexico,capilene 2 fabric has an open knit that invite...
497,498,all-time shell,no need to use that morning times as an umbrel...,[2.5-layer 100% all-recycled polyester shell w...,"Shell: 2.5-layer, 4.1-oz 50-denier 100% all-re...",493.0,17.4,China,2.5-layer 100% all-recycled polyester shell wi...
498,499,all-wear cargo shorts,all-wear cargo shorts bask in the glory of swe...,"[lightweight organic cotton canvas, zip fly wi...",5.2-oz 100% organic cotton canvas. Recyclable ...,349.0,12.1,Thailand,lightweight organic cotton canvas | zip fly wi...


In [69]:
# remplacement des abreviations et contractions

import re
import pandas as pd

# Abréviation -> mot complet. Valeur "" = pas encore défini : la clé est ignorée (voir rapport_abreviations)
DICO_MOTS = {
    "jkt": "jacket", "btm": "bottom", "ctn": "cotton", "p/o": "pullover", "pkt": "pocket",
    "fz": "full-zip", "reg": "regular", "b.c.": "British Columbia",
    "S/s": "short sleeve", "L/s": "long sleeve",
    "Lw": "lightweight", "Mw": "midweight", "Hw": "heavyweight", "Ulw": "ultralight",
    "pfd": "personal flotation device",
    "R1": "", "R2": "", "r4": "", "M10": "", "Mlc": "",
}

import re

CONTRACTIONS = {
    "won't": "will not", "can't": "can not", "shan't": "shall not", "ain't": "is not",
    "n't": " not", "'re": " are", "'ve": " have", "'ll": " will", "'d": " would", "'m": " am",
    "let's": "let us", "it's": "it is", "that's": "that is", "there's": "there is",
    "here's": "here is", "what's": "what is", "who's": "who is", "he's": "he is", "she's": "she is",
}
_RE_SPECIAUX = re.compile(r"\b(" + "|".join(re.escape(k) for k in CONTRACTIONS if "'" in k and not k.startswith("'") and k != "n't") + r")\b", re.I)
_RE_SUFFIXES = re.compile(r"(n't|'re|'ve|'ll|'d|'m)\b", re.I)
_RE_POSSESSIF = re.compile(r"\b([A-Za-z]+)'s\b")


def etendre_contractions(texte):
    """won't -> will not ; you're -> you are ; girl's -> girl (possessif retiré)."""
    if not isinstance(texte, str):
        return texte
    texte = texte.replace("’", "'")                                  # apostrophe courbe -> droite
    texte = _RE_SPECIAUX.sub(lambda m: CONTRACTIONS[m.group(1).lower()], texte)   # formes irrégulières
    texte = _RE_SUFFIXES.sub(lambda m: CONTRACTIONS[m.group(1).lower()], texte)   # don't, you're...
    texte = _RE_POSSESSIF.sub(r"\1", texte)                          # girl's -> girl
    return re.sub(r"\s+", " ", texte).strip()




# Cas contextuels : "in." seulement après un chiffre ; " (pouce) seulement après un chiffre ou une fraction
SPECIAUX = [
    (re.compile(r"(?<=\d)\s*in\.(?!\w)", re.I), " inch"),
    (re.compile(r"(?<=[\d/½¼¾])\s*\"+"), " inch"),
]


def compiler_dico(dico: dict) -> list[tuple[re.Pattern, str]]:
    """Regex par clé (insensible à la casse, mot entier), clés longues d'abord. Ignore les valeurs vides."""
    regs = []
    for cle in sorted(dico, key=len, reverse=True):
        if not dico[cle].strip():
            continue
        regs.append((re.compile(rf"(?<![\w/]){re.escape(cle)}(?![\w/])", re.I), dico[cle]))
    return regs


def remplacer_abreviations(texte, regs):
    if not isinstance(texte, str):
        return texte
    for pat, rep in SPECIAUX + regs:
        texte = pat.sub(rep, texte)
    return re.sub(r"\s+", " ", texte).strip()


def appliquer_dico(df: pd.DataFrame, dico: dict = DICO_MOTS, cols=("name", "text", "details", "fabric")) -> pd.DataFrame:
    """Copie de df avec abréviations remplacées (colonnes texte ou listes de textes)."""
    regs = compiler_dico(dico)
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            raise KeyError(f"Colonne absente : {c}")
        out[c] = out[c].map(lambda v: [remplacer_abreviations(x, regs) for x in v] if isinstance(v, list)
                            else remplacer_abreviations(v, regs))
    return out


def rapport_abreviations(df: pd.DataFrame, dico: dict = DICO_MOTS, cols=("name", "text", "details", "fabric")) -> pd.DataFrame:
    """Occurrences de chaque clé du dico dans le corpus, et si elle est définie."""
    textes = pd.concat([df[c].explode() if df[c].map(lambda v: isinstance(v, list)).any() else df[c]
                        for c in cols]).dropna().astype(str)
    lignes = []
    for cle, val in dico.items():
        pat = re.compile(rf"(?<![\w/]){re.escape(cle)}(?![\w/])", re.I)
        lignes.append({"cle": cle, "remplacement": val or "(non défini)", "occurrences": int(textes.str.count(pat).sum())})
    return pd.DataFrame(lignes).sort_values("occurrences", ascending=False).reset_index(drop=True)


# Remplacement des cntractions 
dataset["text"] = dataset["text"].map(etendre_contractions)
# Remplacement des abreviations
print(rapport_abreviations(dataset))
dataset = appliquer_dico(dataset)

     cle               remplacement  occurrences
0    jkt                     jacket           45
1     R1               (non défini)           43
2     Lw                lightweight           16
3    S/s               short sleeve           13
4    reg                    regular           12
5     R2               (non défini)           10
6    L/s                long sleeve           10
7    btm                     bottom            9
8    Mlc               (non défini)            7
9    ctn                     cotton            6
10   M10               (non défini)            6
11    r4               (non défini)            6
12   p/o                   pullover            4
13   pfd  personal flotation device            2
14  b.c.           British Columbia            2
15    fz                   full-zip            1
16    Mw                  midweight            1
17   pkt                     pocket            1
18    Hw                heavyweight            1
19   Ulw            

In [70]:
dataset.head()

,id,name,text,details,fabric,weight_g,weight_oz,origin,details_txt
0,1,active classic boxers,there is a reason why our boxers are a cult fa...,"[silky capilene 1 fabric is ultralight, breath...",3.7-oz 100% all-recycled polyester with Gladio...,99.0,3.5,Mexico,"silky capilene 1 fabric is ultralight, breatha..."
1,2,active sport boxer briefs,skinning up glory requires enough movement wit...,"[stretch mesh provides support, open-weave mes...",4.6-oz 93% polyester (71% recycled)/7% spandex...,60.0,2.1,Israel,"stretch mesh provides support, open-weave mesh..."
2,3,active sport briefs,these superbreathable no-fly briefs are the mi...,"[stretch mesh provides support, open-weave mes...",4.6-oz 93% polyester (71% recycled)/7% spandex...,49.0,1.7,Israel,"stretch mesh provides support, open-weave mesh..."
3,4,alpine guide pants,"skin in, climb ice, switch to rock, traverse a...",[durable stretch-woven polyester (47% recycled...,7.6-oz 90-denier 92% polyester (47% recycled)/...,597.0,20.7,Vietnam,durable stretch-woven polyester (47% recycled)...
4,5,alpine wind jacket,"on high ridges, steep ice and anything alpine,...","[lightweight, breathable polyester ripstop fab...",Shell: 1.3-oz 20-denier 100% polyester ripstop...,331.0,11.5,China,"lightweight, breathable polyester ripstop fabr..."


In [73]:
#je vais tokenizer ce datraframe, quelle est la meilleure strategie : juste text ou toutes les colonnes
dataset["description"] = dataset["name"] +" " + dataset["text"]  

# clean description by removing special caracters 

dataset["description_clean"] = (
    dataset['description']
    .str.lower()
    .str.replace(r"[^a-z0-9]+"," ",regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
# replace abreviations by real words : 

import en_core_web_sm
nlp = en_core_web_sm.load()

#dataset.head()

# tokenize : 
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from spacy.lang.en.stop_words import STOP_WORDS

#notice : [front, back, top, bottom, side, full, made] sont retirés. Ils sont à la fois vides et descriptifs pour un catalogue de vêtements (back pocket, full zip, top, bottom). Si ces mots comptent pour le clustering, retire-les de la liste.
UTILES = {"front", "back", "top", "bottom", "side", "full", "made", "first", "show", "not", "no"}

MOTS_VIDES = (ENGLISH_STOP_WORDS - UTILES) | {"oz", "recycled", "inch", "recyclable", "program", "common", "threads"}

dataset['description_tokenized'] = (
    dataset['description_clean']
    .apply(lambda x: nlp(x))
    .apply(lambda doc: [token.lemma_ for token in doc if token.text not in MOTS_VIDES])
)

# vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.7, sublinear_tf=True,
#                       stop_words=list(MOTS_VIDES),
#                       token_pattern=r"(?u)\b[a-z][a-z]+\b")     # lettres seulement : plus de nombres
dataset.head()

,id,name,text,details,fabric,weight_g,weight_oz,origin,details_txt,description,description_clean,description_tokenized
0,1,active classic boxers,there is a reason why our boxers are a cult fa...,"[silky capilene 1 fabric is ultralight, breath...",3.7-oz 100% all-recycled polyester with Gladio...,99.0,3.5,Mexico,"silky capilene 1 fabric is ultralight, breatha...",active classic boxers there is a reason why ou...,active classic boxers there is a reason why ou...,"[active, classic, boxer, reason, boxer, cult, ..."
1,2,active sport boxer briefs,skinning up glory requires enough movement wit...,"[stretch mesh provides support, open-weave mes...",4.6-oz 93% polyester (71% recycled)/7% spandex...,60.0,2.1,Israel,"stretch mesh provides support, open-weave mesh...",active sport boxer briefs skinning up glory re...,active sport boxer briefs skinning up glory re...,"[active, sport, boxer, brief, skin, glory, req..."
2,3,active sport briefs,these superbreathable no-fly briefs are the mi...,"[stretch mesh provides support, open-weave mes...",4.6-oz 93% polyester (71% recycled)/7% spandex...,49.0,1.7,Israel,"stretch mesh provides support, open-weave mesh...",active sport briefs these superbreathable no-f...,active sport briefs these superbreathable no f...,"[active, sport, brief, superbreathable, no, fl..."
3,4,alpine guide pants,"skin in, climb ice, switch to rock, traverse a...",[durable stretch-woven polyester (47% recycled...,7.6-oz 90-denier 92% polyester (47% recycled)/...,597.0,20.7,Vietnam,durable stretch-woven polyester (47% recycled)...,"alpine guide pants skin in, climb ice, switch ...",alpine guide pants skin in climb ice switch to...,"[alpine, guide, pant, skin, climb, ice, switch..."
4,5,alpine wind jacket,"on high ridges, steep ice and anything alpine,...","[lightweight, breathable polyester ripstop fab...",Shell: 1.3-oz 20-denier 100% polyester ripstop...,331.0,11.5,China,"lightweight, breathable polyester ripstop fabr...","alpine wind jacket on high ridges, steep ice a...",alpine wind jacket on high ridges steep ice an...,"[alpine, wind, jacket, high, ridge, steep, ice..."


In [74]:
# Detokenize the tokenized sentences and store them in an ```nlp_ready``` column
dataset['nlp_ready']= dataset['description_tokenized'].apply(lambda x: ' '.join(x))
dataset.head()

,id,name,text,details,fabric,weight_g,weight_oz,origin,details_txt,description,description_clean,description_tokenized,nlp_ready
0,1,active classic boxers,there is a reason why our boxers are a cult fa...,"[silky capilene 1 fabric is ultralight, breath...",3.7-oz 100% all-recycled polyester with Gladio...,99.0,3.5,Mexico,"silky capilene 1 fabric is ultralight, breatha...",active classic boxers there is a reason why ou...,active classic boxers there is a reason why ou...,"[active, classic, boxer, reason, boxer, cult, ...",active classic boxer reason boxer cult favorit...
1,2,active sport boxer briefs,skinning up glory requires enough movement wit...,"[stretch mesh provides support, open-weave mes...",4.6-oz 93% polyester (71% recycled)/7% spandex...,60.0,2.1,Israel,"stretch mesh provides support, open-weave mesh...",active sport boxer briefs skinning up glory re...,active sport boxer briefs skinning up glory re...,"[active, sport, boxer, brief, skin, glory, req...",active sport boxer brief skin glory require mo...
2,3,active sport briefs,these superbreathable no-fly briefs are the mi...,"[stretch mesh provides support, open-weave mes...",4.6-oz 93% polyester (71% recycled)/7% spandex...,49.0,1.7,Israel,"stretch mesh provides support, open-weave mesh...",active sport briefs these superbreathable no-f...,active sport briefs these superbreathable no f...,"[active, sport, brief, superbreathable, no, fl...",active sport brief superbreathable no fly brie...
3,4,alpine guide pants,"skin in, climb ice, switch to rock, traverse a...",[durable stretch-woven polyester (47% recycled...,7.6-oz 90-denier 92% polyester (47% recycled)/...,597.0,20.7,Vietnam,durable stretch-woven polyester (47% recycled)...,"alpine guide pants skin in, climb ice, switch ...",alpine guide pants skin in climb ice switch to...,"[alpine, guide, pant, skin, climb, ice, switch...",alpine guide pant skin climb ice switch rock t...
4,5,alpine wind jacket,"on high ridges, steep ice and anything alpine,...","[lightweight, breathable polyester ripstop fab...",Shell: 1.3-oz 20-denier 100% polyester ripstop...,331.0,11.5,China,"lightweight, breathable polyester ripstop fabr...","alpine wind jacket on high ridges, steep ice a...",alpine wind jacket on high ridges steep ice an...,"[alpine, wind, jacket, high, ridge, steep, ice...",alpine wind jacket high ridge steep ice alpine...


In [75]:
# Use sklearn to calculate the tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy

# TruncatedSVD from sklearn
from sklearn.decomposition import TruncatedSVD

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.7, sublinear_tf=True,
                       stop_words=list(MOTS_VIDES),
                       token_pattern=r"(?u)\b[a-z][a-z]+\b")     # lettres seulement : plus de nombres
X = vectorizer.fit_transform(dataset['nlp_ready'])
dense = X.toarray()
dense

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(500, 5808))

In [76]:
# explore vocabulary : 
vectorizer.vocabulary_

{'active': 28,
 'classic': 723,
 'boxer': 459,
 'cult': 1084,
 'favorite': 1644,
 'cool': 947,
 'sticky': 4762,
 'situation': 4441,
 'quick': 3779,
 'dry': 1294,
 'lightweight': 2716,
 'underwear': 5303,
 'minimal': 3052,
 'space': 4644,
 'travel': 5209,
 'pack': 3346,
 'expose': 1535,
 'brush': 520,
 'waistband': 5454,
 'offer': 3275,
 'skin': 4465,
 'softness': 4619,
 'panel': 3380,
 'construction': 918,
 'traditional': 5186,
 'back': 243,
 'fit': 1733,
 'functional': 1954,
 'fly': 1823,
 'make': 2887,
 'polyester': 3606,
 'moisture': 3081,
 'wicke': 5638,
 'performance': 3460,
 'inseam': 2392,
 'size': 4442,
 'recycle': 3853,
 'active classic': 30,
 'quick dry': 3781,
 'travel pack': 5212,
 'skin softness': 4482,
 'panel construction': 3381,
 'functional fly': 1955,
 'fly make': 1834,
 'make polyester': 2917,
 'polyester moisture': 3626,
 'moisture wicke': 3092,
 'wicke performance': 5642,
 'inseam size': 2395,
 'size recycle': 4448,
 'sport': 4689,
 'brief': 510,
 'glory': 2003,
 '

In [77]:
svd_model = TruncatedSVD(n_components=50,n_iter = 100 , random_state=122)
lsa = svd_model.fit_transform(X)

topic_encoded_df = pd.DataFrame(lsa, columns = [f"topic_{i+1}" for i in range(50)])
# topic_encoded_df["text_clean"] = dataset['description_clean']
topic_encoded_df.index = dataset.index
topic_encoded_df

,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,...,topic_41,topic_42,topic_43,topic_44,topic_45,topic_46,topic_47,topic_48,topic_49,topic_50
0,0.200571,-0.075578,0.057782,0.011798,-0.020220,0.085255,-0.034628,-0.086620,-0.024207,-0.034149,...,0.185695,0.051912,-0.068443,0.081903,0.022335,-0.037720,-0.109863,-0.018942,0.042163,-0.007883
1,0.188739,-0.066963,0.028546,-0.008838,-0.026446,0.090532,0.025326,-0.086457,-0.092329,-0.028921,...,0.155003,0.017342,-0.015167,0.112777,0.014916,-0.059052,-0.064689,0.059208,-0.025264,-0.069914
2,0.191073,-0.088727,0.021895,-0.007291,-0.054918,0.047585,0.091650,-0.060212,-0.022165,-0.027504,...,0.117673,-0.010079,-0.033329,0.066056,-0.029329,-0.063867,-0.058095,0.044788,-0.087255,-0.080493
3,0.162854,-0.099168,-0.063272,-0.002020,0.002193,-0.062313,-0.021663,0.044404,-0.132602,-0.069385,...,0.070399,0.011656,0.014752,-0.109429,0.155843,-0.142209,-0.072955,0.037615,0.069320,-0.136017
4,0.176493,-0.106421,-0.045422,0.001504,-0.056140,-0.134064,-0.023120,0.095302,-0.093355,0.019715,...,0.192167,-0.094202,0.029649,0.061340,0.108817,-0.012570,0.021716,-0.060060,0.208003,-0.054433
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.230666,-0.108517,0.181475,-0.012909,-0.330050,0.146474,-0.127508,-0.061824,-0.032157,-0.162067,...,0.024869,0.002197,-0.026636,0.127672,0.047629,0.048651,0.147421,0.119449,0.063171,0.028329
496,0.244624,-0.095915,0.263936,-0.012394,-0.373795,0.103872,-0.113417,-0.017442,0.042622,-0.075502,...,-0.010603,0.049996,-0.022573,-0.002549,-0.016224,0.069245,0.007902,0.000845,-0.078222,-0.079168
497,0.198335,-0.119662,-0.067319,-0.011730,-0.024167,-0.187442,-0.035255,0.110816,-0.046136,-0.016023,...,0.078162,-0.009774,-0.028240,0.032361,-0.032561,-0.055470,0.016767,0.019742,-0.009549,-0.080707
498,0.200708,-0.019281,-0.041416,-0.005266,0.140864,0.174626,-0.167857,-0.042457,-0.038031,-0.035408,...,-0.016800,-0.021941,-0.084275,0.053588,-0.021168,0.102468,-0.056925,0.060774,0.021774,-0.008142


In [52]:
topic_encoded_df['name'] = dataset.name
topic_encoded_df

,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,...,topic_42,topic_43,topic_44,topic_45,topic_46,topic_47,topic_48,topic_49,topic_50,name
0,0.384812,-0.039436,0.145084,0.117335,-0.108459,-0.042426,-0.071018,-0.016325,-0.148988,-0.043958,...,-0.075542,0.010720,0.013289,0.044447,-0.041203,-0.008921,0.017721,-0.104761,-0.043214,active classic boxers
1,0.387946,-0.058814,0.074514,0.118295,-0.017122,0.063695,-0.094795,-0.045732,-0.078663,-0.117631,...,-0.069896,0.040001,-0.060678,0.039935,-0.022605,0.016247,0.047446,-0.070954,0.029980,active sport boxer briefs
2,0.380640,-0.070148,0.099663,0.046045,-0.022095,0.112034,-0.026089,0.028852,-0.039076,-0.048421,...,-0.017626,0.011927,-0.062467,0.081517,0.055044,0.007420,0.048367,-0.059766,0.093311,active sport briefs
3,0.470576,-0.243566,-0.069401,-0.168060,0.034352,-0.050620,-0.001694,-0.063040,0.004369,-0.168023,...,-0.065325,-0.022451,0.017604,-0.029938,-0.073706,-0.070855,-0.015971,-0.009661,-0.033933,alpine guide pants
4,0.441213,-0.260541,0.020050,-0.242027,-0.039085,-0.051868,0.027450,0.035040,0.089861,-0.166386,...,-0.029722,-0.079337,0.005881,-0.029737,0.067682,-0.015189,0.084162,-0.062892,0.060503,alpine wind jacket
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.430351,-0.067341,0.356391,0.106579,-0.246168,-0.083308,-0.051272,-0.062058,-0.323512,-0.078348,...,-0.009053,-0.033361,0.080930,-0.054479,0.032030,-0.064763,0.026536,-0.024141,-0.002560,cap 2 bottoms
496,0.432299,-0.056355,0.461814,0.040906,-0.303156,-0.085413,-0.026329,0.026083,-0.259645,0.028752,...,0.000583,0.017412,0.022165,-0.042031,0.009663,0.016471,0.032973,0.014901,0.000402,cap 2 crew
497,0.382063,-0.210675,-0.018609,-0.271783,-0.010977,-0.143233,0.161444,-0.136625,0.006348,0.062440,...,-0.060904,-0.034291,0.028958,-0.038047,-0.051552,0.017747,0.048027,0.019772,-0.035608,all-time shell
498,0.457553,0.039410,-0.171487,0.281413,0.031659,-0.164480,-0.069882,0.015453,-0.068447,0.008265,...,0.030265,0.032040,-0.009337,0.001947,-0.058845,0.005481,-0.005602,-0.072977,-0.020129,all-wear cargo shorts


In [82]:
svd_model.explained_variance_ratio_.sum()

np.float64(0.3508808677129156)

In [79]:
import numpy as np
variance_cumulee =  np.cumsum(svd_model.explained_variance_ratio_)

In [80]:
import plotly_express as px

px.line(variance_cumulee)

On va rechercher entre 10 et 20 clusters dans notre catalogue décrit dans la matrice TF-IDF
Pour cela on utilise DBSCAN avec une metrique cosine, pour rechercher des articles proches au sens du corpus des tokens de la description. et un eps à ajuster  
     
Pour plus de pertinence, j'ai choisi de prendre notre matrice TF-IDF décomposée en LSA sur un nb de composantes entre 20 et 100.  LSA va permettre de décrire l'espace des mots des descriptions autour d'axes rassemblant les composantes principales décrivant les tokens, au lieu de se noyer dans 500 dimensions creuses du catalogue.  
  
On recherche combien de dimensions de LSA permettent de bien décrire notre catalogue tout en minimisant le bruit, et en assurant un plus gros cluster pas trop gros. 
On fait pour cela varier : 
- le nb de dimensions entre 20 et 75 
- eps_grid au départ entre 0.15 et 0.8 puis entre 0.05 et 0.40 par pas de 0.0125
- min_samples parmi : 2, 3, 4, 5,6,    


In [83]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer


def scanner_dimensions(X, dims=( 30, 37, 45,  50, 57,60, 67, 70), cible=20, bruit_max=0.20,
                       eps_grid=None, min_samples_grid=(2, 3, 4), seed=42) -> pd.DataFrame:
    """Pour chaque nb de composantes LSA : meilleur (eps, min_samples) proche de `cible` clusters."""
    eps_grid = np.arange(0.15, 0.80, 0.0125) if eps_grid is None else eps_grid
    n_max = min(max(dims), X.shape[1] - 1, X.shape[0] - 1)
    Zfull = TruncatedSVD(n_components=n_max, random_state=seed).fit_transform(X)

    lignes = []
    for k in (d for d in dims if d <= n_max):
        Z = Normalizer().fit_transform(Zfull[:, :k])             # renormaliser après troncature
        meilleur = None
        for ms in min_samples_grid:
            for eps in eps_grid:
                lab = DBSCAN(eps=float(eps), min_samples=ms, metric="cosine").fit_predict(Z)
                ok = lab != -1
                if ok.sum() == 0:
                    continue
                tailles = np.bincount(lab[ok])
                n, bruit = len(tailles), 1 - ok.mean()
                if bruit > bruit_max:
                    continue
                # on privilégie : proche de la cible, puis cluster dominant petit, puis peu de bruit
                cle = (abs(n - cible), tailles.max() / len(Z), bruit)
                if meilleur is None or cle < meilleur[0]:
                    meilleur = (cle, dict(n_comp=k, eps=round(float(eps), 3), min_samples=ms,
                                          n_clusters=n, bruit=round(bruit, 3),
                                          plus_gros=round(tailles.max() / len(Z), 2)))
        if meilleur:
            lignes.append(meilleur[1])
    return pd.DataFrame(lignes)

res = scanner_dimensions(X,# descend sous 0,15
    min_samples_grid=(2, 3, 4, 5,6),
)           # X = matrice TF-IDF (lemmatisée)
print(res.to_string(index=False))


 n_comp   eps  min_samples  n_clusters  bruit  plus_gros
     30 0.200            3          20  0.040       0.47
     37 0.225            6          20  0.180       0.14
     45 0.300            4          20  0.040       0.54
     50 0.313            4          20  0.066       0.44
     57 0.388            2          20  0.004       0.81
     60 0.363            4          20  0.078       0.45
     67 0.463            2          20  0.000       0.84
     70 0.375            5          20  0.186       0.27


In [ ]:

res = scanner_dimensions(
    X, dims=(24, 26, 27, 28, 29, 30, 31,32, 33, 37, 40, 43, 46),
    eps_grid=np.arange(0.05, 0.40, 0.0125),     # descend sous 0,15
    min_samples_grid=(2, 3, 4, 5,6),
)           # X = matrice TF-IDF (lemmatisée)
print(res.to_string(index=False))

La meilleure solution : 30 composantes, qui permet de minimiser le bruit ! 
30 composantes permettent d'expliquer 38% de la variance 
  
Affichons maintenant les nuages de mots des 20 clusters : 



In [59]:
# Etude de sensibilite sur 30 composantes : 
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import Normalizer

N_MAX, SEED = 46, 42                       # = max(dims) du dernier scan
CANDIDATS = {29:(0.137,3) ,30: (0.1375, 5), 33: (0.1625, 5), 43: (0.2, 5)}   # n_comp: (eps, min_samples)

Zfull = TruncatedSVD(n_components=N_MAX, random_state=SEED).fit_transform(X)


def clusteriser(k, eps, ms):
    Z = Normalizer().fit_transform(Zfull[:, :k])
    return Z, DBSCAN(eps=eps, min_samples=ms, metric="cosine").fit_predict(Z)


def resume(lab) -> str:
    s = pd.Series(lab[lab != -1]).value_counts()
    return f"{len(s)} clusters | bruit {np.mean(lab == -1):.1%} | plus gros {s.iloc[0] / len(lab):.0%}"

def rattacher_bruit(Z: np.ndarray, lab: np.ndarray) -> np.ndarray:
    """Affecte chaque point de bruit (-1) au centroïde de cluster le plus proche (cosinus)."""
    lab = lab.copy()
    ids = np.unique(lab[lab != -1])
    if len(ids) == 0 or not (lab == -1).any():
        return lab
    C = np.vstack([Z[lab == k].mean(axis=0) for k in ids])
    C /= np.linalg.norm(C, axis=1, keepdims=True)
    lab[lab == -1] = ids[np.argmax(Z[lab == -1] @ C.T, axis=1)]
    return lab

def vectoriser(textes_lem, min_df=2, max_df=0.7, ngram=(1, 2)):
    """TF-IDF (norme L2, sous-linéaire) sur les textes déjà lemmatisés."""
    vec = TfidfVectorizer(ngram_range=ngram, min_df=min_df, max_df=max_df,
                          sublinear_tf=True, token_pattern=r"(?u)\b[a-z][a-z]+\b")
    X = vec.fit_transform(textes_lem)
    vide = int((X.getnnz(axis=1) == 0).sum())
    if vide:
        print(f"⚠️ {vide} texte(s) sans aucun terme retenu (vecteur nul)")
    return vec, X

def termes_par_cluster(X, labels, vec, top=8) -> pd.DataFrame:
    """Termes TF-IDF les plus forts du centroïde de chaque cluster."""
    termes = np.array(vec.get_feature_names_out())
    lignes = []
    for k in np.unique(labels):
        centre = np.asarray(X[labels == k].mean(axis=0)).ravel()
        lignes.append({"cluster": k, "n": int((labels == k).sum()),
                       "termes": ", ".join(termes[np.argsort(-centre)[:top]])})
    return pd.DataFrame(lignes).set_index("cluster")



res = {k: clusteriser(k, *p) for k, p in CANDIDATS.items()}
for k, (Z, lab) in res.items():
    print(f"{k} comp. : {resume(lab)}")

# Les candidats s'accordent-ils ? (ARI proche de 1 = même partition)
cles = list(res)
ari = pd.DataFrame([[adjusted_rand_score(res[a][1], res[b][1]) for b in cles] for a in cles],
                   index=cles, columns=cles).round(2)
print(ari)

# Partition retenue
Z, lab = res[30]
dataset["cluster_dbscan"] = lab                           # -1 = bruit
dataset["cluster_final"] = rattacher_bruit(Z, lab)        # option : rattacher le bruit au plus proche
print(termes_par_cluster(X, lab, vectorizer).to_string())
print(dataset[dataset["cluster_dbscan"] == -1][["id", "name"]].to_string(index=False))   # le bruit

29 comp. : 20 clusters | bruit 1.2% | plus gros 31%
30 comp. : 20 clusters | bruit 4.2% | plus gros 29%
33 comp. : 20 clusters | bruit 4.8% | plus gros 28%
43 comp. : 20 clusters | bruit 8.6% | plus gros 28%
      29    30    33    43
29  1.00  0.85  0.76  0.78
30  0.85  1.00  0.84  0.78
33  0.76  0.84  1.00  0.80
43  0.78  0.78  0.80  1.00
           n                                                                 termes
cluster                                                                            
-1        21                        li, br, spandex, nylon, web, buckle, inch, roll
 0       102                     li, br, pocket, dwr, finish, water, nylon, durable
 1        33                 li, pocket, br, deni, strap, mesh, polyurethane, water
 2        20                         li, 82, br, 18, coverage, nylon, spandex, line
 3       145           br, li, organic, cotton, shirt, recycle, recyclable, program
 4        10                br, li, headband, brim, glare, underbill,

In [ ]:
# pip install wordcloud matplotlib
import math
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud


def poids_cluster(X, labels, vec, k, top=60) -> dict:
    """Poids TF-IDF moyens des `top` termes du cluster k (centroïde)."""
    termes = np.asarray(vec.get_feature_names_out())
    centre = np.asarray(X[labels == k].mean(axis=0)).ravel()
    idx = np.argsort(-centre)[:top]
    return {termes[i]: float(centre[i]) for i in idx if centre[i] > 0}


def wordclouds_clusters(X, labels, vec, top=60, colonnes=4, inclure_bruit=False, fichier=None,
                        min_points=1, cmap="viridis"):
    """Une wordcloud par cluster (taille = poids TF-IDF moyen du cluster), en grille."""
    labels = np.asarray(labels)
    if X.shape[0] != len(labels):
        raise ValueError(f"X a {X.shape[0]} lignes, labels {len(labels)}")
    ids = [k for k in np.unique(labels) if (inclure_bruit or k != -1) and (labels == k).sum() >= min_points]
    if not ids:
        raise ValueError("Aucun cluster à afficher")

    lignes = math.ceil(len(ids) / colonnes)
    fig, axes = plt.subplots(lignes, colonnes, figsize=(4.5 * colonnes, 3.2 * lignes), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, k in zip(axes.ravel(), ids):
        poids = poids_cluster(X, labels, vec, k, top)
        if poids:
            wc = WordCloud(width=600, height=400, background_color="white", colormap=cmap,
                           prefer_horizontal=0.9, random_state=42).generate_from_frequencies(poids)
            ax.imshow(wc, interpolation="bilinear")
        titre = "Bruit" if k == -1 else f"Cluster {k}"
        ax.set_title(f"{titre} (n={int((labels == k).sum())})", fontsize=11)
    fig.tight_layout()
    if fichier:
        fig.savefig(fichier, dpi=150, bbox_inches="tight")
    return fig


# Usage : X et vec = matrice et TfidfVectorizer ; labels = tes clusters DBSCAN
fig = wordclouds_clusters(X, dataset["cluster_dbscan"].to_numpy(), vec, fichier="wordclouds.png")
plt.show()




Ça dépend de ta tâche, mais en général : un seul texte composite (name, text, details, fabric), avec les colonnes numériques et catégorielles gardées à part.

Ce que je recommande

Tokenizer : name, text, details et fabric. Ils portent le sens du produit. Mets name en premier, pour qu’une troncature ne le coupe jamais.
Ne pas tokenizer weight_g, weight_oz et origin : ce sont des features structurées (numérique, catégorielle). Dans un tokenizer, 340 devient des sous-mots sans valeur numérique.
Retirer le boilerplate : « Recyclable through the Common Threads Recycling Program » est dans fabric de 351 lignes sur 500. Ça gaspille des tokens sans information.
Éviter la fuite de cible : si tu prédis origin ou le poids, exclus-les du texte. Ici origin est de toute façon dans la description d’origine (« Made in … »).
Dédoublonner avant le split : 66 noms sont dupliqués (même produit, variantes de couleur). Sinon train et test partagent des quasi-doublons.

je veux lemmatize le texte et ensuite TFIDF transformation avec tfidfVectorizer pour train un clustering model et regrouper les produits aux descriptions similaires

Pipeline en 4 étapes : lemmatisation spaCy, TfidfVectorizer, réduction LSA (TruncatedSVD) puis KMeans. Le texte vient de construire_texte (boilerplate déjà retiré).

Points d’attention :

Silhouette faible : sur du texte TF-IDF, 0,05 à 0,2 est normal. Juge surtout la cohérence des termes_par_cluster et des noms regroupés.
Chiffres supprimés : t.is_alpha écarte les nombres (3.7-oz, Cap 3), donc « Cap 1/2/3/4 » sont proches. Pour les distinguer, garde les tokens alphanumériques (t.is_alpha or t.like_num), ou ajoute le poids comme feature numérique à côté.
Vocabulaire métier : si un terme domine tous les clusters (par exemple fabric, oz), passe-le dans mots_exclus=frozenset({...}), ou baisse max_df.
Fit sur tout le jeu : le clustering est non supervisé, donc ajuster le TF-IDF sur tous les produits est correct. Si tu évalues ensuite un modèle supervisé, ajuste-le sur le train uniquement.
Alternatives : AgglomerativeClustering(metric="cosine", linkage="average") ne demande pas de choisir k à l’avance si tu fixes distance_threshold. HDBSCAN (sklearn.cluster.HDBSCAN) détecte aussi les produits « isolés ».
Compatibilité : tout tourne sur CPU, sous Windows comme sur Apple Silicon. Aucun tenseur torch n’est utilisé.

In [ ]:
# pip install spacy scikit-learn
# python -m spacy download en_core_web_sm      (descriptions en anglais)
import numpy as np
import pandas as pd
import spacy
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import Normalizer


def charger_spacy(modele: str = "en_core_web_sm"):
    """Charge spaCy sans parser ni NER (inutiles ici, donc plus rapide)."""
    try:
        return spacy.load(modele, disable=["parser", "ner"])
    except OSError as e:
        raise OSError(f"Modèle absent. Lance : python -m spacy download {modele}") from e


def lemmatiser(textes, nlp, mots_exclus: frozenset = frozenset()) -> list[str]:
    """Lemmes alphabétiques en minuscules, sans stop words ni mots exclus."""
    sortie = []
    # n_process=1 : le multiprocessing de spaCy est fragile sous Windows / notebook
    for doc in nlp.pipe((str(t) for t in textes), batch_size=64):
        lemmes = [t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop and len(t) > 1]
        sortie.append(" ".join(l for l in lemmes if l not in mots_exclus))
    return sortie


def vectoriser(textes_lem, min_df=2, max_df=0.7, ngram=(1, 2)):
    """TF-IDF (norme L2, sous-linéaire) sur les textes déjà lemmatisés."""
    vec = TfidfVectorizer(ngram_range=ngram, min_df=min_df, max_df=max_df,
                          sublinear_tf=True, token_pattern=r"(?u)\b[a-z][a-z]+\b")
    X = vec.fit_transform(textes_lem)
    vide = int((X.getnnz(axis=1) == 0).sum())
    if vide:
        print(f"⚠️ {vide} texte(s) sans aucun terme retenu (vecteur nul)")
    return vec, X


def reduire_lsa(X, n_comp=100, seed=42):
    """LSA + normalisation L2 : KMeans euclidien ≈ distance cosinus."""
    n = max(2, min(n_comp, X.shape[1] - 1, X.shape[0] - 1))
    Z = TruncatedSVD(n_components=n, random_state=seed).fit_transform(X)
    return Normalizer(copy=False).fit_transform(Z)


def choisir_k(Z, ks=range(5, 31, 5), seed=42) -> tuple[int, pd.Series]:
    """Meilleur k selon le silhouette (cosinus)."""
    scores = {}
    for k in ks:
        if k >= len(Z):
            break
        lab = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(Z)
        scores[k] = silhouette_score(Z, lab, metric="cosine")
    s = pd.Series(scores, name="silhouette")
    return int(s.idxmax()), s


def termes_par_cluster(X, labels, vec, top=8) -> pd.DataFrame:
    """Termes TF-IDF les plus forts du centroïde de chaque cluster."""
    termes = np.array(vec.get_feature_names_out())
    lignes = []
    for k in np.unique(labels):
        centre = np.asarray(X[labels == k].mean(axis=0)).ravel()
        lignes.append({"cluster": k, "n": int((labels == k).sum()),
                       "termes": ", ".join(termes[np.argsort(-centre)[:top]])})
    return pd.DataFrame(lignes).set_index("cluster")


# ---------- Usage ----------
nlp = charger_spacy()
dataset["texte"] = construire_texte(dataset)
dataset["texte_lem"] = lemmatiser(dataset["texte"], nlp)

vec, X = vectoriser(dataset["texte_lem"])
Z = reduire_lsa(X)

k, scores = choisir_k(Z)
print(scores.round(3).to_string(), f"\n-> k = {k}")

dataset["cluster"] = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Z)
print(termes_par_cluster(X, dataset["cluster"].to_numpy(), vec).to_string())
print(dataset.groupby("cluster")["name"].apply(lambda s: ", ".join(s.head(5))).to_string())


# Bonus : produits les plus proches d'un produit donné (cosinus)
def voisins(i: int, Z, df, n=5) -> pd.DataFrame:
    sim = Z @ Z[i]                       # Z est normalisée : produit scalaire = cosinus
    idx = np.argsort(-sim)[1:n + 1]
    return df.iloc[idx][["id", "name"]].assign(similarite=sim[idx].round(3))

5     0.147
10    0.194
15    0.237
20    0.270
25    0.292
30    0.334 
-> k = 30
          n                                                                                                      termes
cluster                                                                                                                
0         4                          tricot line, tricot, zippered fly, cuff position, weave polyester, pant, hip, cuff
1        55                           tap shoulder, tap, shoulder seam, ringspun, shirt, phthalate free, phthalate, ink
2        17                                       dress, jersey knit, knee, knit, flattering, jersey, bodice, fall knee
3        12                                  supplex nylon, supplex, oz supplex, ply oz, ply, nylon, baggies, nylon dwr
4        26                                inch, inseam, cotton canvas, canvas, button, pocket, button closure, zip fly
5        21                               fleece, polyester, polyester fleece

In [ ]:
import spacy
spacy.__version__

'3.8.16'

In [ ]:
!python -m spacy download en_core_news_sm 


✘ No compatible package found for 'en_core_news_sm' (spaCy v3.8.16)



In [ ]:
!python -m spacy download en_core_web_sm 

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 8.5 MB/s eta 0:00:02
     -------- ------------------------------- 2.6/12.8 MB 8.9 MB/s eta 0:00:02
     ---------------- ----------------------- 5.2/12.8 MB 10.3 MB/s eta 0:00:01
     ------------------------- -------------- 8.1/12.8 MB 11.2 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 11.3 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 11.5 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [ ]:
# let's create a complete corpus : 
# Import english stop words
from spacy.lang.en.stop_words import STOP_WORDS

In [ ]:
print(STOP_WORDS)

{'she', 'mostly', 'but', 'side', 'two', 'may', 'above', 'move', 'along', '’re', 'once', 'anyway', 'quite', 'around', 'first', 'each', 'show', 're', '’ve', 'sometimes', 'through', 'every', 'thence', 'thru', 'latterly', 'anywhere', 'twelve', 'of', 'last', 'might', 'via', 'mine', 'done', 'again', 'itself', 'under', 'unless', 'upon', 'former', 'go', 'him', 'beforehand', 'another', 'your', 'into', 'hereafter', 'back', '‘s', 'both', 'per', 'thereby', 'various', 'off', 'least', 'twenty', 'serious', 'thereupon', 'whereupon', 'namely', 'other', 'becomes', 'still', 'also', 'they', '‘re', 'any', 'does', "'m", "'ve", 'full', 'between', 'cannot', 'forty', '‘m', 'the', 'hence', 'out', 'keep', 'thereafter', 'nobody', 'ever', 'same', 'herein', 'we', 'wherein', 'four', 'very', 'would', 'anyhow', 'most', 'regarding', 'from', 'had', 'together', 'up', 'even', 'until', 'amount', 'whence', '‘ve', 'yourself', 'therefore', 'either', "n't", 'yourselves', 'therein', 'about', 'if', 'put', 'eight', 'not', 'top', 

In [ ]:
# 500 articles with Id  | text : nom article - description 
# Baggies shorts - Even Baggies, our most popular shorts for anything, or nothing, occasionally need an update. This season we've increased the inseam length. Their casual fit, quick-drying water-repellent nylon and elasticized waistband with an internal drawstring remain the same as ever. Other features include a polyester mesh lining, a rear snap pocket and front pockets (with self-draining mesh corners) positioned to reduce drag in the water. Inseam (size M) is 7"". Recyclable through the Common Threads Recycling Program.<br><br><b>Details:</b><ul> <li>Quick-drying nylon with a DWR (durable water repellent) finish</li> <li>Elasticized waistband with internal drawstring; black mesh liner</li> <li>Vertical on-seam side pockets for reduced drag in the water; pocket bags have quick-drain-and-dry mesh corners; snap-closed back pocket</li> <li>""Inseam is 7""""""</li></ul><br><br><b>Fabric: </b>4.2-oz 100% nylon with a DWR finish. Lining: 5.2-oz 100% polyester mesh. Recyclable through the Common Threads Recycling Program<br><br><b>Weight: </b>(233 g 8.1 oz)<br><br>Made in Nicaragua."
# Tokenizer : name, text, details et fabric
# lowercase caracters of description
dataset["description_clean"] = (
    dataset['description']
    .str.lower()
    .str.replace(r"[^a-z0-9]+"," ",regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
dataset.head()

KeyError: 'description'

In [ ]:
# create object nlp:
import en_core_web_sm
nlp = en_core_web_sm.load()

In [ ]:
# tokenize: 
# Tokenize the cleaned sentences and remove english stopwords
from spacy.lang.en.stop_words import STOP_WORDS
dataset['description_tokenized'] = (
    dataset['description_clean']
    .apply(lambda x: nlp(x))
    .apply(lambda doc: [token.lemma_ for token in doc if token.text not in STOP_WORDS])
)
dataset.head()

,id,description,description_,name,category,article,description_clean,description_tokenized
0,1,There's a reason why our boxers are a cult fav...,Active classic boxers - There's a reason why o...,Active classic boxers,Active,classic boxers,there s a reason why our boxers are a cult fav...,"[s, reason, boxer, cult, favorite, cool, espec..."
1,2,Skinning up Glory requires enough movement wit...,Active sport boxer briefs - Skinning up Glory ...,Active sport boxer briefs,Active,sport boxer briefs,skinning up glory requires enough movement wit...,"[skin, glory, require, movement, boxer, decide..."
2,3,These superbreathable no-fly briefs are the mi...,Active sport briefs - These superbreathable no...,Active sport briefs,Active,sport briefs,these superbreathable no fly briefs are the mi...,"[superbreathable, fly, brief, minimalist, s, c..."
3,4,"Skin in, climb ice, switch to rock, traverse a...","Alpine guide pants - Skin in, climb ice, switc...",Alpine guide pants,Alpine,guide pants,skin in climb ice switch to rock traverse a kn...,"[skin, climb, ice, switch, rock, traverse, kni..."
4,5,"On high ridges, steep ice and anything alpine,...","Alpine wind jkt - On high ridges, steep ice an...",Alpine wind jkt,Alpine,wind jkt,on high ridges steep ice and anything alpine t...,"[high, ridge, steep, ice, alpine, jacket, serv..."


In [ ]:
# Detokenize the tokenized sentences and store them in an ```nlp_ready``` column
dataset['nlp_ready']= dataset['description_tokenized'].apply(lambda x: ' '.join(x))
dataset.head()

,id,description,description_,name,category,article,description_clean,description_tokenized,nlp_ready
0,1,There's a reason why our boxers are a cult fav...,Active classic boxers - There's a reason why o...,Active classic boxers,Active,classic boxers,there s a reason why our boxers are a cult fav...,"[s, reason, boxer, cult, favorite, cool, espec...",s reason boxer cult favorite cool especially s...
1,2,Skinning up Glory requires enough movement wit...,Active sport boxer briefs - Skinning up Glory ...,Active sport boxer briefs,Active,sport boxer briefs,skinning up glory requires enough movement wit...,"[skin, glory, require, movement, boxer, decide...",skin glory require movement boxer decide poach...
2,3,These superbreathable no-fly briefs are the mi...,Active sport briefs - These superbreathable no...,Active sport briefs,Active,sport briefs,these superbreathable no fly briefs are the mi...,"[superbreathable, fly, brief, minimalist, s, c...",superbreathable fly brief minimalist s choice ...
3,4,"Skin in, climb ice, switch to rock, traverse a...","Alpine guide pants - Skin in, climb ice, switc...",Alpine guide pants,Alpine,guide pants,skin in climb ice switch to rock traverse a kn...,"[skin, climb, ice, switch, rock, traverse, kni...",skin climb ice switch rock traverse knife edge...
4,5,"On high ridges, steep ice and anything alpine,...","Alpine wind jkt - On high ridges, steep ice an...",Alpine wind jkt,Alpine,wind jkt,on high ridges steep ice and anything alpine t...,"[high, ridge, steep, ice, alpine, jacket, serv...",high ridge steep ice alpine jacket serve true ...


In [ ]:
#Use sklearn to calculate the tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy

# TruncatedSVD from sklearn
from sklearn.decomposition import TruncatedSVD

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(dataset['nlp_ready'])
dense = X.toarray()
dense

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(500, 3807))

In [ ]:
vectorizer.vocabulary_

{'reason': 2656,
 'boxer': 623,
 'cult': 1006,
 'favorite': 1372,
 'cool': 932,
 'especially': 1301,
 'sticky': 3175,
 'situation': 2998,
 'quick': 2613,
 'dry': 1191,
 'lightweight': 1994,
 'underwear': 3549,
 'minimal': 2167,
 'space': 3100,
 'travel': 3468,
 'pack': 2370,
 'expose': 1332,
 'brush': 654,
 'waistband': 3661,
 'offer': 2309,
 'skin': 3010,
 'softness': 3080,
 'panel': 2388,
 'construction': 910,
 'traditional': 3451,
 'classic': 798,
 'fit': 1415,
 'functional': 1517,
 'fly': 1457,
 'oz': 2367,
 '100': 3,
 'recycled': 2666,
 'polyester': 2511,
 'moisture': 2190,
 'wicke': 3737,
 'performance': 2441,
 'inseam': 1823,
 'size': 2999,
 'recyclable': 2664,
 'common': 866,
 'thread': 3389,
 'recycle': 2665,
 'program': 2567,
 'br': 625,
 'details': 1098,
 'ul': 3525,
 'li': 1983,
 'silky': 2977,
 'capilene': 709,
 'fabric': 1349,
 'ultralight': 3530,
 'breathable': 632,
 'elastic': 1235,
 'comfort': 857,
 'gladiodor': 1559,
 'natural': 2237,
 'odor': 2308,
 'control': 923,
 

# Part 1 : Groups of products with similar descriptions

For this part, you'll have to train a **clustering model** that will create groups of products for which the descriptions are "close" to each other. 
- You can use `DBSCAN` to make some clustering on the TF-IDF matrix. When dealing with texts, the distance metric to be used is "cosine" instead of "euclidean". You have to find some values for `eps` and `min_samples` that allow to get 10-20 clusters and not too many outliers.
- You can display a `wordcloud` for each cluster to analyze the results and see if the groups contain different words.
